<a href="https://colab.research.google.com/github/Anyaporwal/Bone-Fracture-Detection/blob/main/A2_20_AnyaPorwal_Practi10_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Name : Anya Porwal
## Roll No. : A2 -20
## Branch/Year : CSE(DS)/3rd yr (section A)
## Course : Deep Learning-1 Lab
## Practical No. 10 (Project Work)

#Bone Fracture Detection - Detects fractures from X-ray images
---
Dataset used : https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data

Model used : CNN model ( that classifies X-ray images into - Fractured/Normal)

---

In [1]:
import os # Used to interact with system environment variables

# Setting Kaggle username as an environment variable
os.environ['KAGGLE_USERNAME'] = "anap12"
# Setting Kaggle API key (token)
os.environ['KAGGLE_KEY'] = "KGAT_aa649f07b8c4d626ee241cfeefe69e58"

In [2]:
!pip install kaggle

In [3]:
# Download Dataset from kaggle
!kaggle datasets download -d bmadushanirodrigo/fracture-multi-region-x-ray-data

Dataset URL: https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data
License(s): ODC Public Domain Dedication and Licence (PDDL)
100% 481M/481M [00:04<00:00, 118MB/s]



In [4]:
!unzip fracture-multi-region-x-ray-data.zip

Streaming output truncated to the last 5000 lines.
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3 (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3-rotated1 (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3-rotated1.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated3 (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/

In [5]:
import os # # Used to interact with the operating system (file paths, environment variables, etc.)
import tensorflow as tf # Deep learning library used to build and train neural networks
from tensorflow.keras import layers, models # layers - used to create CNN layers (Conv2D, Dense, etc.)
# models - used to build the overall neural network architecture
import matplotlib.pyplot as plt # Used for plotting graphs (like training accuracy and loss)

In [6]:
img_size = 224 # Resize all images to 224x224 pixels (standard size for CNN models)
batch_size = 32 # # Number of images processed at once during training

# Load training dataset from directory
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train",
    image_size=(img_size, img_size),
    batch_size=batch_size
)

# Load validation dataset (used to evaluate model performance)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/val",
    image_size=(img_size, img_size),
    batch_size=batch_size
)

class_names = train_ds.class_names

Found 9246 files belonging to 2 classes.
Found 829 files belonging to 2 classes.


In [7]:
from tensorflow.keras import layers
import tensorflow as tf

# Normalize images
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# Ignore corrupted images (ADD HERE)
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [8]:
print("Classes:", class_names)

Classes: ['fractured', 'not fractured']


In [9]:
from tensorflow.keras import models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # Binary output
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
import tensorflow as tf

train_ds = train_ds.apply(tf.data.experimental.ignore_errors()).repeat()
val_ds = val_ds.apply(tf.data.experimental.ignore_errors()).repeat()

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3
    327/Unknown 1047s 3s/step - accuracy: 0.7566 - loss: 0.4992

In [ ]:
loss, acc = model.evaluate(val_ds)
print("Validation Accuracy:", acc)

20/20 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.5875 - loss: 0.6930
Validation Accuracy: 0.5874999761581421


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/test.jpg"  # upload your test image

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("Fracture Detected")
else:
    print("Normal Bone")